In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
data_dir = "/content/drive/MyDrive"

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

import matplotlib.pyplot as plt

from PIL import Image
import torchvision
from torchvision import transforms, datasets

from torch.utils.data import (
    DataLoader,
    random_split
)

In [ ]:
import shutil

if not os.path.exists("/content/dataset_split"):

    shutil.copytree(
        "/content/drive/MyDrive/dataset_split",
        "/content/dataset_split"
    )

print("Dataset skopiowany lokalnie")

Dataset skopiowany lokalnie


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])



train_dir = "/content/dataset_split/train"
val_dir = "/content/dataset_split/val"
test_dir = "/content/dataset_split/test"

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=val_dir,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

# klasy
class_names = train_dataset.classes

print("Klasy:")
print(class_names)

print(f"\nTrain images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

# **ResNet18**

In [ ]:
device = torch.device("cuda")

from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(
    model.fc.in_features,
    len(train_dataset.classes)
)

model = model.to(device)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

#for param in model.layer4.parameters():
    #param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True


batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
#resnet18 stage1-5
optimizer = optim.Adam( model.fc.parameters(),lr=6e-5,weight_decay=1e-4)

#resnet50
#optimizer = optim.Adam([
    #{'params': model.layer4.parameters(), 'lr': 3e-6},
    #{'params': model.fc.parameters(),     'lr': 6e-5}
#])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

In [ ]:
print(type(model))
print(model)

<class 'torchvision.models.resnet.ResNet'>
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stat

In [ ]:
num_epochs = 20
best_val_accuracy = 0.0

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    if epoch == 5:

        print("Odblokowuję layer4")

        for param in model.layer4.parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {'params': model.layer4.parameters(), 'lr': 2e-5},
            {'params': model.fc.parameters(),     'lr': 5e-5}
        ],weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
    )
    if epoch == 12:

        print("Odblokowuję całość ")

        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.layer1.parameters(), 'lr': 5e-7},
            {'params': model.layer2.parameters(), 'lr': 5e-7},
            {'params': model.layer3.parameters(), 'lr': 1e-6},
            {'params': model.layer4.parameters(), 'lr': 2e-6},
            {'params': model.fc.parameters(),     'lr': 1e-5}
        ])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
    )

    print(f"\nEpoka {epoch+1}/{num_epochs}")



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val acc:   {val_accuracy:.4f}"
    )



    torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/runs_resnet18_stage2_1/model_current_resnet18.pth"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/runs_resnet18_stage2_1/model_best_resnet18.pth"
        )

        print("Zapisano najlepszy model")

    scheduler.step(val_accuracy)

print(f"Best accuracy: {best_val_accuracy:.4f}")


Epoka 1/20
Train loss: 2.5386 | Train acc: 0.2372
Val loss:   2.2692 | Val acc:   0.3347
Zapisano najlepszy model

Epoka 2/20
Train loss: 1.9487 | Train acc: 0.4846
Val loss:   1.9066 | Val acc:   0.4504
Zapisano najlepszy model

Epoka 3/20
Train loss: 1.6164 | Train acc: 0.5923
Val loss:   1.6723 | Val acc:   0.5529
Zapisano najlepszy model

Epoka 4/20
Train loss: 1.4119 | Train acc: 0.6486
Val loss:   1.4906 | Val acc:   0.5924
Zapisano najlepszy model

Epoka 5/20
Train loss: 1.2667 | Train acc: 0.6811
Val loss:   1.4155 | Val acc:   0.6253
Zapisano najlepszy model
Odblokowuję layer4

Epoka 6/20
Train loss: 0.6159 | Train acc: 0.8172
Val loss:   0.5738 | Val acc:   0.8022
Zapisano najlepszy model

Epoka 7/20
Train loss: 0.3408 | Train acc: 0.8838
Val loss:   0.3991 | Val acc:   0.8458
Zapisano najlepszy model

Epoka 8/20
Train loss: 0.2741 | Train acc: 0.9015
Val loss:   0.4057 | Val acc:   0.8381

Epoka 9/20
Train loss: 0.2365 | Train acc: 0.9119
Val loss:   0.4486 | Val acc:   0.8

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/runs_resnet18_stage2_1/model_best_resnet18.pth"
    )
)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

In [ ]:
runs_dir = "/content/drive/MyDrive/runs_resnet18_stage2_1"
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(true_labels, pred_labels)

precision = precision_score(
    true_labels,
    pred_labels,
    average='macro'
)

recall = recall_score(
    true_labels,
    pred_labels,
    average='macro'
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average='macro'
)

print("\nWyniki modelu")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

metrics_file = os.path.join(
    runs_dir,
    "test_metrics.txt"
)

with open(metrics_file, "w") as f:

    f.write("Wyniki modelu\n")
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1-score : {f1:.4f}\n")

print(f"\nZapisano do: {metrics_file}")


Wyniki modelu
Accuracy : 0.8861
Precision: 0.8933
Recall   : 0.8824
F1-score : 0.8835

Zapisano do: /content/drive/MyDrive/runs_resnet18_stage2_1/test_metrics.txt


In [ ]:
import seaborn as sns

from sklearn.metrics import confusion_matrix
runs_dir = "/content/drive/MyDrive/runs_resnet18_stage2_1"


os.makedirs(runs_dir, exist_ok=True)

# loss plot

plt.figure(figsize=(10,5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')

plt.title("Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "loss_plot.png"),
    bbox_inches='tight'
)

plt.close()

# accuracy plot

plt.figure(figsize=(10,5))

plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Val Accuracy')

plt.title("Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "accuracy_plot.png"),
    bbox_inches='tight'
)

plt.close()

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    true_labels,
    pred_labels
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_dataset.classes,
    yticklabels=test_dataset.classes
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.savefig(
    os.path.join(
        runs_dir,
        "test_confusion_matrix.png"
    ),
    bbox_inches="tight"
)

plt.close()

print("\nZapisano:")
print("- model_best.pth")
print("- model_current.pth")
print("- loss_plot.png")
print("- accuracy_plot.png")
print("- confusion_matrix.png")


Zapisano:
- model_best.pth
- model_current.pth
- loss_plot.png
- accuracy_plot.png
- confusion_matrix.png


#Metryki dla klas

In [23]:
!cp -r "/content/drive/MyDrive/dataset_split/test" "/content/test_local"

In [24]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

test_dir_metryki = "/content/drive/MyDrive/dataset_split/test"


test_dataset = datasets.ImageFolder(
    "/content/test_local",
    transform=transform
)
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [25]:
device = torch.device("cuda")

from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(
    model.fc.in_features,
    len(test_dataset.classes)
)

model = model.to(device)

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/runs_resnet18_stage2/model_best_resnet18.pth",
        map_location=device
    )
)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

In [27]:
from sklearn.metrics import classification_report

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=test_dataset.classes,
        digits=4
    )
)

              precision    recall  f1-score   support

     APSK128     1.0000    0.9463    0.9724       205
      APSK16     0.9958    0.9634    0.9793       246
      APSK32     0.9337    0.9683    0.9506       189
      APSK64     0.9588    0.9749    0.9668       239
        ASK4     1.0000    0.9600    0.9796       225
        ASK8     0.9784    1.0000    0.9891       227
        BPSK     0.8000    0.7895    0.7947       228
        GMSK     0.9652    0.8622    0.9108       225
       PSK16     0.7043    0.8044    0.7510       225
       PSK32     0.8895    0.7156    0.7931       225
        PSK8     0.9355    0.7665    0.8426       227
      QAM128     0.9864    0.8821    0.9313       246
       QAM16     0.9646    0.9959    0.9800       246
      QAM256     0.8704    0.9553    0.9109       246
       QAM32     0.9598    0.9637    0.9618       248
       QAM64     0.9153    0.9153    0.9153       248
        QPSK     0.5431    0.7456    0.6285       228

    accuracy              